# 02 — Run the scrape

Menjalankan pipeline dari notebook, dengan progress bar. Cocok untuk run kecil
dan untuk memantau apa yang sedang terjadi.

Untuk run besar, pakai CLI — lebih hemat memori dan tidak ikut mati kalau kernel
di-restart:

```powershell
python main.py search
python main.py enrich
python main.py images
python main.py export --format jsonl csv parquet
```

Semua sel di bawah memanggil fungsi dari `src/` — tidak ada logika pipeline yang
ditulis ulang di sini. Kalau perilakunya perlu berubah, ubah `src/`, bukan notebook.

**Aman diinterupsi.** Database di-commit setiap halaman dan setiap produk, jadi
`Interrupt Kernel` tidak menghilangkan apa pun. Jalankan ulang untuk melanjutkan.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from tokopedia_scraper.config import Config
from tokopedia_scraper.logging_setup import setup_logging
from tokopedia_scraper.storage import Storage

cfg = Config.load(ROOT / "config.yaml")
cfg.ensure_dirs()
setup_logging("INFO", cfg.logging.file, force=True)

print(f"fetcher   : {cfg.fetcher}")
print(f"keywords  : {len(cfg.keywords)}")
print(f"pages/kw  : {cfg.search.max_pages_per_keyword}   rows/page: {cfg.search.rows_per_page}")
print(f"delay     : {cfg.rate_limit.min_delay}-{cfg.rate_limit.max_delay}s, concurrency {cfg.rate_limit.concurrency}")
print(f"db        : {cfg.storage.db_path}")

In [ ]:
# Progress helper, wired to pipeline's progress(done, total, label) callback.
from IPython.display import clear_output


def notebook_progress(prefix):
    def report(done, total, label):
        clear_output(wait=True)
        pct = done / max(total, 1)
        bar = "#" * int(pct * 40)
        print(f"{prefix}: [{bar:<40}] {done}/{total}  {label}")

    return report


def show(stats, title):
    print(f"\n{title}")
    for key, value in stats.as_dict().items():
        if value:
            print(f"  {key:<18}{value}")
    for note in stats.notes[:10]:
        print(f"  ! {note}")
    if len(stats.notes) > 10:
        print(f"  ... {len(stats.notes) - 10} catatan lain, lihat {cfg.logging.file}")

## Kondisi saat ini

Jalankan sel ini kapan saja untuk melihat posisi. Aman dijalankan sementara
scraping berjalan di terminal lain — SQLite dalam mode WAL, pembaca tidak
memblokir penulis.

In [ ]:
with Storage(cfg.storage.db_path) as store:
    for key, value in store.stats().items():
        print(f"{key:<22}{value:>8,}")

## Stage 1 — kumpulkan daftar produk

Keyword yang sudah selesai dilewati tanpa satu pun request. Keyword yang terputus
dilanjutkan dari halaman berikutnya.

Mulai kecil: satu keyword, dua halaman. Naikkan setelah yakin.

In [ ]:
from tokopedia_scraper.fetchers.base import get_fetcher
from tokopedia_scraper.pipeline import run_search

KEYWORDS = cfg.keywords[:1]   # None -> semua keyword di config.yaml
MAX_PAGES = 2                 # None -> cfg.search.max_pages_per_keyword

with Storage(cfg.storage.db_path) as store:
    fetcher = get_fetcher(cfg)
    try:
        stats = run_search(
            cfg, store, fetcher, KEYWORDS,
            max_pages=MAX_PAGES,
            progress=notebook_progress("search"),
        )
    finally:
        fetcher.close()

show(stats, "stage 1 selesai")

## Stage 2 — deskripsi dari halaman produk

Hanya produk dengan `pdp_fetched = 0` yang diproses, jadi ini otomatis resume.
Satu request per produk — dengan jeda 2–5 detik, 1.000 produk memakan sekitar
satu jam.

In [ ]:
from tokopedia_scraper.pipeline import run_enrich

LIMIT = 10   # None -> semua yang pending

with Storage(cfg.storage.db_path) as store:
    print(f"pending: {store.stats()['pending_pdp']:,}")
    fetcher = get_fetcher(cfg)
    try:
        stats = run_enrich(
            cfg, store, fetcher, limit=LIMIT, progress=notebook_progress("enrich")
        )
    finally:
        fetcher.close()

show(stats, "stage 2 selesai")

## Gambar

URL galeri dari PDP tidak bertanda tangan dan tidak kedaluwarsa, jadi tahap ini
boleh dijalankan kapan saja setelah `enrich`.

Beda halnya untuk produk yang belum di-enrich: thumbnail dari hasil search
**bertanda tangan dan mati dalam beberapa jam**. Urutan yang benar tetap
`enrich` dulu, baru `images`.

In [ ]:
from tokopedia_scraper.pipeline import run_images

with Storage(cfg.storage.db_path) as store:
    stats = run_images(cfg, store, limit=10, progress=notebook_progress("images"))

show(stats, "gambar selesai")
print(f"\ntersimpan di: {cfg.images.dir}")

## Re-parse (tanpa jaringan)

Kalau parser diperbaiki setelah data terlanjur terkumpul, sel ini membangun ulang
tabel `products` dari response mentah yang tersimpan. Nol request.

In [ ]:
from tokopedia_scraper.pipeline import reparse_from_raw

RUN_REPARSE = False   # ubah ke True kalau parser berubah

if RUN_REPARSE:
    with Storage(cfg.storage.db_path) as store:
        stats = reparse_from_raw(cfg, store, progress=notebook_progress("reparse"))
    show(stats, "reparse selesai")
else:
    print("dilewati (RUN_REPARSE = False)")

## Ekspor

In [ ]:
from tokopedia_scraper.pipeline import export_dataset

with Storage(cfg.storage.db_path) as store:
    paths = export_dataset(cfg, store, ("jsonl", "csv", "parquet"))

for path in paths:
    print(f"{path.name:<22}{path.stat().st_size:>12,} bytes")

print("\nLanjut ke 03_eda_dataset.ipynb untuk menilai kualitasnya.")